# 2026 U.S. Senate Midterm Model (Starter Version)

This notebook builds a probabilistic forecast of the 2026 Senate elections in six steps:

1. **Load and clean** state polls, partisan leans, and national approval polls
2. **Weight polls** (recency, sample size, LV/RV, pollster quality, partisan sponsor, generic-ballot polls, pollster flooding) into a state polling average
3. **Estimate the national environment** from presidential approval
4. **Build a fundamentals prior** for every race (`2 × PVI + national environment`) and **combine it with the polls** using a Bayesian (normal-normal) update
5. **Simulate each state independently** to get P(D win) per race
6. **Simulate the whole map with correlated errors** (national + regional + partisan-lean similarity) to get a seat distribution and the chance of each party controlling the Senate

All margins are **Democrat (or credible opposition) minus Republican**, in percentage points.

Each section starts with a **Tuning** cell. Those values are modeling assumptions you can argue about and change.

## Step 0: Setup

Imports, file locations, the simulation settings, and the Senate map.

Put the three data files (`leans_and_regions.csv`, `Midterm_Polls.xlsx`, `National_Polls.xlsx`) in the same folder as this notebook, or edit the paths below.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

pd.set_option("display.width", 200)

In [4]:
# Notebooks don't have __file__, so we use the folder the notebook is running from
LEANS_FILE = ""
POLLS_FILE = ""
NATIONAL_FILE = ""
OUT_DIR = ""
#Include any other datasets you might have like inflation or something else entirely

ELECTION_DAY = pd.Timestamp("2026-11-03")
TODAY = pd.Timestamp.today().normalize()   # This will make sure your script knows when today is
N_SIMS = 10_000
SEED = 2026

rng = np.random.default_rng(SEED)

### The Senate map

Which seats are up in 2026 and which party currently holds each

`HOLDOVER_D` / `HOLDOVER_R` are the seats *not* on the ballot. A 50-50 Senate goes to Republicans via the Vice President's tiebreaking vote, so Democrats need 51. You can make a fair argument that they might need 52 if Fetterman switches in PA - but operate on the 51 assumption unless we get more data to suggest otherwise.

In [5]:
SEATS_UP = {
    # Republican-held
    "Alabama": "R", "Alaska": "R", "Arkansas": "R", "Florida": "R", "Idaho": "R",
    "Iowa": "R", "Kansas": "R", "Kentucky": "R", "Louisiana": "R", "Maine": "R",
    "Mississippi": "R", "Montana": "R", "Nebraska": "R", "North Carolina": "R",
    "Ohio": "R", "Oklahoma": "R", "South Carolina": "R", "South Dakota": "R",
    "Tennessee": "R", "Texas": "R", "West Virginia": "R", "Wyoming": "R",
    # Democratic-held
    "Colorado": "D", "Delaware": "D", "Georgia": "D", "Illinois": "D",
    "Massachusetts": "D", "Michigan": "D", "Minnesota": "D", "New Hampshire": "D",
    "New Jersey": "D", "New Mexico": "D", "Oregon": "D", "Rhode Island": "D",
    "Virginia": "D",
}
HOLDOVER_D = 34   # seats NOT up in 2026 (Dems + the two Indies who caucus with them)
HOLDOVER_R = 31
D_SEATS_FOR_CONTROL = 51   # 50-50 goes to R via the VP tiebreak

# Races where the "Dem%" column is actually an independent (per the Sheet2 note).
# Value = probability that independent caucuses with Democrats if they win. Will/would he? I give it 80%...hence the 0.8 value
#YOU MAY WISH TO CHANGE THIS ASSUMPTION
INDEPENDENT_CAUCUS_PROB_D = {"Nebraska": 0.8}

print(f"{len(SEATS_UP)} races; holdovers D {HOLDOVER_D}, R {HOLDOVER_R}; "
      f"D needs {D_SEATS_FOR_CONTROL - HOLDOVER_D} wins for control")

35 races; holdovers D 34, R 31; D needs 17 wins for control


## Step 1: Load and clean the data

Real data is messy. Before modeling anything, check for:

- **Polls for states that don't have a race** this cycle

The partisan lean is a Cook-style PVI: positive = leans Democratic, negative = leans Republican.

In [6]:
def load_leans(path):
    leans = pd.read_csv(path)
    leans = leans[leans["State"] != "Source"].copy()          # drop citation row
    leans["State"] = leans["State"].str.strip()
    leans["PartisanLean"] = pd.to_numeric(leans["PartisanLean"])
    return leans.set_index("State")


def load_polls(path):
    p = pd.read_excel(path, sheet_name="Sheet1")
    p["State"] = p["State"].str.strip()
    p["LVorRV"] = p["LVorRV"].str.upper().str.strip()
    p["Generic"] = p["Generic"].eq("Yes")
    p["Bigger_Field"] = p["Bigger_Field"].eq("Yes")
    p["Partisan"] = p["Partisan (Pollster or Sponsor)"]
    p["margin"] = p["Dem%"] - p["Rep%"]
    unknown = set(p["State"]) - set(SEATS_UP)
    if unknown:
        print(f"WARNING: polls for states without a race: {unknown}")
    return p


def load_national(path):
    n = pd.read_excel(path, sheet_name="Sheet1")
    n["net"] = n["Approval"] - n["Disapproval"]
    return n

In [7]:
leans = load_leans(LEANS_FILE)
polls_raw = load_polls(POLLS_FILE)
nat = load_national(NATIONAL_FILE)

print(f"{len(leans)} states with leans, {len(polls_raw)} state polls, {len(nat)} national polls")
missing_lean = set(SEATS_UP) - set(leans.index)
if missing_lean:
    print(f"WARNING: races with no partisan lean: {missing_lean}")
polls_raw.head()

FileNotFoundError: [Errno 2] No such file or directory: ''

## Step 2: Weight the polls and build state polling averages

Not all polls are equally informative. Each poll gets a weight that is the **product** of several factors:

| Factor | Idea |
|---|---|
| Recency | Older polls decay with a half-life |
| Sample size | Bigger samples count more, but with diminishing returns (square root) |
| LV vs RV | Likely-voter polls are closer to the actual electorate |
| Partisan sponsor | Polls paid for by a party get down-weighted *and* shifted |
| Generic ballot | "Generic D vs generic R" isn't the real matchup |
| Bigger field | Full-field polls vs head-to-heads (a debatable assumption!) |
| Pollster quality | Based on VoteHub rank |
| Accuracy | Pollsters with smaller historical error count more |
| Flooding | Stops one prolific pollster from dominating a state |

We also **adjust the margin** of some polls (house effects): RV polls and partisan polls get nudged.

Then each state's average comes with an uncertainty that has two parts: random error that shrinks with more polls, and a **systematic error** that doesn't (all polls in a state can miss together).

### ⚙️ Tuning knobs: poll weighting

In [ ]:
RECENCY_HALF_LIFE_DAYS = #     # I think 21 is a good starting point, but that's based on my own priors. I think that after 3 weeks, I expect things to change a bit
REFERENCE_SAMPLE = #           # sample-size weight = sqrt(n / REFERENCE_SAMPLE) ----- What do you think the baseline sample size is?
RV_WEIGHT = #                  # pollsters think these people are more likely to vote, so forecasts often give them more credibility. But it also gives more opportunity to put your thumb on the scale
RV_MARGIN_SHIFT = #          # RV polls skew slightly D; nudge toward R ---- there is conflicting evidence about this and many don't think this is accurate in 2026
PARTISAN_WEIGHT = #
PARTISAN_HOUSE_EFFECT = #     # shift sponsor-party polls this many pts against sponsor ---- is this the right amount? I don't know
GENERIC_WEIGHT = #           # "generic D vs generic R" polls are weaker signal...they do not force someone to actually pick a candidate. Soemtimes people vote for candidates not of their own party!
# Bigger_Field = "Yes": poll lists every ballot candidate (or the two majors + "not sure").
# My assumption: head-to-head polls force a choice and reveal truer preferences,
# so full-field polls count for less.
BIGGER_FIELD_WEIGHT = #

## Pollster quality score ("Pollster Rank" column): 5 = lowest quality, 10 = highest. I made this scale and it isn't the cleanest, an A+ on Votehub is a 10, an A is a 9, and so forth
## If there is no rating, it gets a 5
# Linear map: score 5 -> MIN_QUALITY_WEIGHT, score 10 -> 1.0
QUALITY_SCORE_MIN = 5
QUALITY_SCORE_MAX = 10
MIN_QUALITY_WEIGHT = 0.5         # weight for the worst-rated pollster; lower = quality matters more
UNRANKED_SCORE = #               # pollsters missing a score are treated as below average
# Typical Error = pollster's average absolute miss on the margin (points).
# For a normal distribution, SD ~= 1.25 * mean absolute error.
MAE_TO_SD = 1.25
DEFAULT_TYPICAL_ERROR = 4.0      # used where Typical Error is blank --- right now this is all of them! You can fill in the blanks.

# Shared polling miss within a state; does NOT shrink with more polls
SYSTEMATIC_POLL_ERROR_SD = 4.5

In [ ]:
def weight_polls(p):
    p = p.copy()
    age = (TODAY - p["Date"]).dt.days.clip(lower=0)
    p["w_recency"] = 0.5 ** (age / RECENCY_HALF_LIFE_DAYS)
    p["w_size"] = np.sqrt(p["SampleSize"] / REFERENCE_SAMPLE)
    p["w_type"] = np.where(p["LVorRV"] == "RV", RV_WEIGHT, 1.0)
    p["w_partisan"] = np.where(p["Partisan"].notna(), PARTISAN_WEIGHT, 1.0)
    p["w_generic"] = np.where(p["Generic"], GENERIC_WEIGHT, 1.0)
    p["w_field"] = np.where(p["Bigger_Field"], BIGGER_FIELD_WEIGHT, 1.0)
    score = p["Pollster Rank"].fillna(UNRANKED_SCORE).clip(QUALITY_SCORE_MIN, QUALITY_SCORE_MAX)
    frac = (score - QUALITY_SCORE_MIN) / (QUALITY_SCORE_MAX - QUALITY_SCORE_MIN)
    p["w_quality"] = MIN_QUALITY_WEIGHT + (1 - MIN_QUALITY_WEIGHT) * frac
    # Each poll's total error SD: the larger of pure sampling error and the
    # pollster's historical error (which already includes sampling error)
    pd_, pr_ = p["Dem%"] / 100, p["Rep%"] / 100
    samp_sd = 100 * np.sqrt((pd_ + pr_ - (pd_ - pr_) ** 2) / p["SampleSize"])
    hist_sd = MAE_TO_SD * p["Typical Error"].fillna(DEFAULT_TYPICAL_ERROR)
    p["poll_error_sd"] = np.maximum(samp_sd, hist_sd)
    # More accurate pollsters also get more weight (inverse variance, relative to default)
    p["w_accuracy"] = (MAE_TO_SD * DEFAULT_TYPICAL_ERROR / p["poll_error_sd"]) ** 2
    # Keep one prolific pollster from dominating a state
    n_by_pollster = p.groupby(["State", "Pollster"])["Pollster"].transform("count")
    p["w_flood"] = 1 / np.sqrt(n_by_pollster)

    p["weight"] = p[[c for c in p.columns if c.startswith("w_")]].prod(axis=1)

    # Margin adjustments (house effects)
    adj = p["margin"].astype(float).copy()
    adj += np.where(p["LVorRV"] == "RV", RV_MARGIN_SHIFT, 0)
    adj -= np.where(p["Partisan"] == "Republican", -PARTISAN_HOUSE_EFFECT, 0)  # R poll -> nudge D
    adj -= np.where(p["Partisan"].isin(["Democrat", "Independent/Dem"]),
                    PARTISAN_HOUSE_EFFECT, 0)                                  # D poll -> nudge R
    p["adj_margin"] = adj
    return p


def polling_averages(p):
    def summarize(g):
        w = g["weight"]
        mean = np.average(g["adj_margin"], weights=w)
        n_eff = w.sum() ** 2 / (w ** 2).sum()                 # Kish effective N
        # Typical single-poll SD on the margin (weighted)
        poll_sd = np.sqrt(np.average(g["poll_error_sd"] ** 2, weights=w))
        sd = np.sqrt(poll_sd ** 2 / n_eff + SYSTEMATIC_POLL_ERROR_SD ** 2)
        return pd.Series({"poll_mean": mean, "poll_sd": sd,
                          "n_polls": len(g), "n_eff": n_eff})
    return p.groupby("State")[["weight", "adj_margin", "poll_error_sd"]] \
            .apply(summarize)

In [ ]:
polls = weight_polls(polls_raw)
poll_avg = polling_averages(polls)

# Peek at how the weights came out for one state (change the state to explore)
weight_cols = [c for c in polls.columns if c.startswith("w_")]
display(polls.loc[polls["State"] == "Maine",
                  ["Pollster", "Date", "margin", "adj_margin"] + weight_cols + ["weight"]]
        .sort_values("weight", ascending=False).round(3))

poll_avg.round(2)

## Step 3: The national environment

Midterms are often a referendum on the President. We turn the President's **net approval** (approve minus disapprove) into a national swing in the Senate margin.

The President is a Republican, so **negative net approval ... probably ... helps Democrats**:

$$\text{env (D margin)} = \text{ENV\_SLOPE} \times -(\text{net approval})$$

National polls are weighted by recency and sample size, and polls of all adults (in my mind) count a bit less than polls of voters.

### ⚙️ Tuning knobs: national environment

In [ ]:
# How many points of Senate margin per point of net approval. Pure assumption: tune it!
#Between 0 and 1 - a larger value will mean a bigger swing toward national environment
ENV_SLOPE = #

In [ ]:
def national_environment(nat):
    age = (TODAY - nat["Date"]).dt.days.clip(lower=0)
    w = 0.5 ** (age / RECENCY_HALF_LIFE_DAYS) * np.sqrt(nat["SampleSize"] / 1000)
    w *= np.where(nat["Type"] == "Adults", 0.8, 1.0)          # adults < RV < LV
    net = np.average(nat["net"], weights=w)
    env = ENV_SLOPE * -net
    print(f"Weighted presidential net approval: {net:+.1f}  ->  "
          f"national environment D{env:+.1f}")
    return env

In [ ]:
env = national_environment(nat)

## Step 4: Fundamentals prior + Bayesian update

**The prior.** Before looking at any state polls, what would we expect? A state's partisan lean plus the national environment:

$$\text{prior margin} = 2 \times \text{PVI} + \text{env}$$

(PVI is measured in vote *share*; a 1-point shift in share is a 2-point shift in *margin*, hence the ×2.)

**Accounting for the Polls.** We combine prior and polls with a precision-weighted average (precision = 1 / variance). Whichever is more certain gets more say. A state with lots of good polls is driven by the polls; a state with none falls back entirely on the prior.

**Time until the election.** Opinion can still move, so we add drift uncertainty that grows with the square root of days remaining.

### ⚙️ Tuning knobs: prior and drift

In [ ]:
PRIOR_SD = #        # how much a race can deviate from lean + environment
DAILY_DRIFT_SD = #   # opinion movement per day until the election

In [ ]:
def build_race_table(leans, poll_avg, env):
    days_left = (ELECTION_DAY - TODAY).days
    drift_sd = DAILY_DRIFT_SD * np.sqrt(days_left)

    races = pd.DataFrame(index=sorted(SEATS_UP))
    races["incumbent_party"] = pd.Series(SEATS_UP)
    races = races.join(leans[["PartisanLean", "Region"]]).join(poll_avg)
    races["prior_mean"] = 2 * races["PartisanLean"] + env    # PVI is in share; x2 for margin
    races["prior_sd"] = PRIOR_SD

    # Normal-normal update: precision-weighted average of prior and polls
    has_polls = races["poll_mean"].notna()
    prec_prior = 1 / races["prior_sd"] ** 2
    prec_poll = np.where(has_polls, 1 / races["poll_sd"] ** 2, 0.0)
    races["post_mean"] = (prec_prior * races["prior_mean"]
                          + prec_poll * races["poll_mean"].fillna(0)) / (prec_prior + prec_poll)
    races["post_sd"] = np.sqrt(1 / (prec_prior + prec_poll) + drift_sd ** 2)
    races["poll_share_of_estimate"] = prec_poll / (prec_prior + prec_poll)
    races["n_polls"] = races["n_polls"].fillna(0).astype(int)
    return races

In [ ]:
races = build_race_table(leans, poll_avg, env)
races[["PartisanLean", "n_polls", "prior_mean", "poll_mean",
       "poll_share_of_estimate", "post_mean", "post_sd"]].sort_values("post_mean").round(2)

## Step 5: State-level simulations (independent)

For each race, draw thousands of possible election-day margins around the posterior estimate and count how often Democrats win.

We use a **Student-t** distribution instead of a normal one. It has fatter tails, so big upsets happen a bit more often, which matches real elections better. The draws are rescaled so the standard deviation stays the same.

These simulations treat every state as **independent**, which is fine for single-race probabilities but wrong for counting seats (see Step 6).

### ⚙️ Tuning knobs: tail thickness

In [ ]:
T_DF = #   # degrees of freedom: lower = fatter tails; set large

In [ ]:
def scaled_t(rng, size, df):
    """Student-t draws rescaled to unit variance."""
    return rng.standard_t(df, size=size) * np.sqrt((df - 2) / df)


def state_simulations(races, rng):
    draws = races["post_mean"].values + races["post_sd"].values * \
        scaled_t(rng, (N_SIMS, len(races)), T_DF)
    races["p_dem_state_sim"] = (draws > 0).mean(axis=0)
    return races

In [ ]:
races = state_simulations(races, rng)
races[["post_mean", "post_sd", "p_dem_state_sim"]].sort_values("post_mean").round(2)

## Step 6: National simulations with correlated errors

Polling misses aren't independent. If polls underestimate Republicans in Iowa, they probably did in Kansas too. To count seats realistically, each simulation draws errors that are **correlated** between states:

$$\text{corr}(i, j) = \text{NAT} + \text{REG} \cdot [\text{same region}] + \text{LEAN} \cdot e^{-|\text{lean}_i - \text{lean}_j| / \text{LEAN\_SCALE}}$$

- **NAT**: every state shares a national component
- **REG**: states in the same region move together more
- **LEAN**: states with similar partisanship move together more

Whatever is left over (`1 - NAT - REG - LEAN`) is state-specific noise.

We also share one "bad polling year" scale factor across all states in a simulation (a multivariate t), so tail events hit everywhere at once. Finally, if a winning independent might caucus with either party, we flip a weighted coin.

### ⚙️ Tuning knobs: correlation structure

In [ ]:
CORR_NATIONAL = #
CORR_REGION = #
CORR_LEAN = #
LEAN_SCALE = #
assert CORR_NATIONAL + CORR_REGION + CORR_LEAN < 1, "correlations must leave room for state noise"

In [ ]:
def correlation_matrix(races):
    lean = races["PartisanLean"].to_numpy(dtype=float)
    region = races["Region"].to_numpy(dtype=object)
    same_region = (region[:, None] == region[None, :]).astype(float)
    lean_sim = np.exp(-np.abs(lean[:, None] - lean[None, :]) / LEAN_SCALE)
    R = CORR_NATIONAL + CORR_REGION * same_region + CORR_LEAN * lean_sim
    np.fill_diagonal(R, 1.0)
    return R


def national_simulations(races, rng):
    k = len(races)
    R = correlation_matrix(races)
    L = np.linalg.cholesky(R)
    z = rng.standard_normal((N_SIMS, k)) @ L.T               # correlated N(0, R)
    # Shared chi-square scale -> multivariate t: "bad polling year" hits every state
    g = rng.chisquare(T_DF, size=(N_SIMS, 1)) / T_DF
    z = z / np.sqrt(g) * np.sqrt((T_DF - 2) / T_DF)
    margins = races["post_mean"].values + races["post_sd"].values * z
    opp_wins = margins > 0                                    # D or credible Ind wins

    # Independents: draw whether each winning Ind caucuses with D
    caucus_d = np.ones_like(opp_wins, dtype=bool)
    for st, prob in INDEPENDENT_CAUCUS_PROB_D.items():
        if st in races.index:
            j = races.index.get_loc(st)
            caucus_d[:, j] = rng.random(N_SIMS) < prob

    d_seats = HOLDOVER_D + (opp_wins & caucus_d).sum(axis=1)
    races["p_dem_national_sim"] = opp_wins.mean(axis=0)
    sims = pd.DataFrame(margins, columns=races.index)
    return races, sims, d_seats

In [ ]:
races, sims, d_seats = national_simulations(races, rng)

print(f"Democratic seats (incl. caucusing Indies): median {np.median(d_seats):.0f}, "
      f"80% interval {np.percentile(d_seats, 10):.0f}-{np.percentile(d_seats, 90):.0f}")
print(f"P(Democratic control, >= {D_SEATS_FOR_CONTROL} seats): "
      f"{(d_seats >= D_SEATS_FOR_CONTROL).mean():.1%}")

## Step 7: Results and reporting

### Race-by-race table

In [ ]:
cols = ["incumbent_party", "PartisanLean", "Region", "n_polls", "poll_mean",
        "prior_mean", "poll_share_of_estimate", "post_mean", "post_sd",
        "p_dem_state_sim", "p_dem_national_sim"]
out = races[cols].sort_values("post_mean")
out.round(2)

### Correlation in action

If Democrats win one state, how does that change their chances in a related state? You can see how it differs with your own code

In [ ]:
def conditional_demo(sims, a, b):
    """Shows the clustering effect: P(D wins b | D wins a) vs unconditional."""
    pa, pb = (sims[a] > 0), (sims[b] > 0)
    print(f"  P(D wins {b}) = {pb.mean():.2f};  "
          f"if D wins {a}: {pb[pa].mean():.2f};  if D loses {a}: {pb[~pa].mean():.2f}")


conditional_demo(sims, "Iowa", "Kansas")
conditional_demo(sims, "Maine", "New Hampshire")
conditional_demo(sims, "Texas", "Alaska")

### Seat distribution chart

A 538-style histogram: blue bars are simulations where Democrats (plus caucusing independents) control the Senate, red bars are Republican control.

In [ ]:
def plot_seat_distribution(d_seats, path=None):
    """538-style histogram of Democratic seats across the national simulations.
    Blue bars = Democratic control (incl. caucusing Indies); red bars = Republican control."""
    DEM_BLUE, REP_RED = "#1A6FB0", "#D7301F"
    seats, counts = np.unique(d_seats, return_counts=True)
    pct = 100 * counts / len(d_seats)
    p_dem = (d_seats >= D_SEATS_FOR_CONTROL).mean()
    mu, sd = d_seats.mean(), d_seats.std()

    with plt.style.context("fivethirtyeight"):
        fig, ax = plt.subplots(figsize=(12, 6.5))
        colors = np.where(seats >= D_SEATS_FOR_CONTROL, DEM_BLUE, REP_RED)
        ax.bar(seats, pct, width=0.85, color=colors, zorder=3)

        # Normal approximation to the simulated distribution
        x = np.linspace(seats.min() - 2, seats.max() + 2, 400)
        ax.plot(x, 100 * norm.pdf(x, mu, sd), color="#3C3C3C", lw=2, ls="--",
                zorder=4, label=f"Normal fit (mean {mu:.1f}, SD {sd:.1f})")

        # Control threshold
        ax.axvline(D_SEATS_FOR_CONTROL - 0.5, color="#3C3C3C", lw=1.5, zorder=2)
        top = pct.max() * 1.15
        ax.text(D_SEATS_FOR_CONTROL - 0.8, top, f"Republican control\n{1 - p_dem:.0%}",
                ha="right", va="top", color=REP_RED, fontsize=15, fontweight="bold")
        ax.text(D_SEATS_FOR_CONTROL - 0.2, top, f"Democratic control\n{p_dem:.0%}",
                ha="left", va="top", color=DEM_BLUE, fontsize=15, fontweight="bold")

        ax.set_ylim(0, top * 1.05)
        ax.set_xlabel("Democratic seats (incl. caucusing independents)")
        ax.set_ylabel("Share of simulations (%)")
        # Top axis shows the same outcome as Republican seats
        sec = ax.secondary_xaxis("top", functions=(lambda v: 100 - v, lambda v: 100 - v))
        sec.set_xlabel("Republican seats")
        ax.legend(loc="upper right", frameon=False, fontsize=11)
        fig.suptitle(f"Senate control: {len(d_seats):,} simulated elections",
                     x=0.02, ha="left", fontsize=20, fontweight="bold")
        fig.text(0.02, 0.01, f"Model run {TODAY:%b %d, %Y}. 50-50 ties go to Republicans "
                 f"(VP tiebreak).", fontsize=10, color="#555555")
        fig.tight_layout(rect=(0, 0.03, 1, 1))
        if path:
            fig.savefig(path, dpi=200, bbox_inches="tight")
        plt.show()
    return fig


plot_seat_distribution(d_seats, Path(OUT_DIR) / "seat_distribution.png");

In [ ]:
# Win probabilities by state (from the correlated national simulations)
def margin_label(m):
    return f"D+{m:.1f}" if m >= 0 else f"R+{-m:.1f}"

def race_rating(p_dem, state):
    opp = "Ind" if state in INDEPENDENT_CAUCUS_PROB_D else "D"
    fav, top = (opp, p_dem) if p_dem >= 0.5 else ("R", 1 - p_dem)
    if top < 0.60:
        return "Toss-up"
    level = "Lean" if top < 0.70 else "Likely" if top < 0.85 else "Safe"


    #PLEASE TWEAK THIS PART - I DONT KNOW IF WE ALL AGREE ON LEAN/LIKELY/SAFE!


    return f"{level} {fav}"

win_table = pd.DataFrame({
    "Incumbent party": races["incumbent_party"],
    "Projected margin": races["post_mean"].apply(margin_label),
    "P(Dem win)": races["p_dem_national_sim"],
    "P(Rep win)": 1 - races["p_dem_national_sim"],
})
win_table["Rating"] = [race_rating(p, st) for st, p in win_table["P(Dem win)"].items()]
# Flag races where the non-Republican candidate is an independent
win_table.index = [f"{st} (Ind)" if st in INDEPENDENT_CAUCUS_PROB_D else st
                   for st in win_table.index]
win_table.index.name = "State"
win_table = win_table.sort_values("P(Dem win)", ascending=False)

win_table.to_csv(Path(OUT_DIR) / "state_win_probabilities.csv")

display_table = win_table.copy()
for col in ["P(Dem win)", "P(Rep win)"]:
    display_table[col] = display_table[col].map("{:.0%}".format)
display_table

### Save outputs

In [ ]:
out.to_csv(OUT_DIR / "race_probabilities.csv")
pd.Series(d_seats).value_counts().sort_index() \
    .rename("n_sims").to_csv(OUT_DIR / "seat_distribution.csv", index_label="dem_seats")
print(f"Saved race_probabilities.csv, seat_distribution.csv, seat_distribution.png to {OUT_DIR}")